In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

BASE_RESULTS = "/content/drive/MyDrive/thesis_intent_corrected code_august2026_crossvalidations(4and5 class)/5cv_5class_intent_12Aug/results"

ORIGINAL_DIR = os.path.join(BASE_RESULTS, "exp1_A_original_only_bert", "BERT-base")
EXPANDED_DIR = os.path.join(BASE_RESULTS, "BERT-base")
CORRECTED_CV1_DIR = os.path.join(BASE_RESULTS, "correctedllm_cv1_fixed_expanded_weights", "BERT-base", "cv1")

OUT_DIR = os.path.join(BASE_RESULTS, "part1_original_vs_expanded_paired_tables")
os.makedirs(OUT_DIR, exist_ok=True)

METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "chitchat_f1",
    "interpret_f1",
    "reveal_f1",
    "revise_f1",
    "inquire_f1",
]

def read_metrics(condition, base_dir, cv):
    path = os.path.join(base_dir, f"cv{cv}", "test_metrics.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    row = df.iloc[0].to_dict()
    row["condition"] = condition
    row["fold"] = f"CV{cv}"
    row["metrics_path"] = path
    return row

# -------------------------
# Load Original and Expanded fold metrics
# -------------------------
rows = []
for cv in range(1, 6):
    rows.append(read_metrics("Original", ORIGINAL_DIR, cv))
    rows.append(read_metrics("Expanded", EXPANDED_DIR, cv))

fold_metrics = pd.DataFrame(rows)
fold_metrics.to_csv(os.path.join(OUT_DIR, "part1_fold_metrics_original_expanded.csv"), index=False)

display(fold_metrics[["condition", "fold"] + METRICS])

# -------------------------
# Mean ± SD summary
# -------------------------
summary_rows = []
for condition, g in fold_metrics.groupby("condition"):
    row = {
        "condition": condition,
        "completed_folds": len(g),
    }
    for m in METRICS:
        mean = g[m].mean()
        sd = g[m].std(ddof=1)
        row[f"{m}_mean"] = mean
        row[f"{m}_sd"] = sd
        row[f"{m}_mean_sd"] = f"{mean:.4f} ± {sd:.4f}"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "part1_5cv_summary_original_expanded.csv"), index=False)

display(summary)

# -------------------------
# Fold-level paired differences: Expanded - Original
# -------------------------
paired_rows = []

for cv in range(1, 6):
    orig = fold_metrics[(fold_metrics["condition"] == "Original") & (fold_metrics["fold"] == f"CV{cv}")].iloc[0]
    exp = fold_metrics[(fold_metrics["condition"] == "Expanded") & (fold_metrics["fold"] == f"CV{cv}")].iloc[0]

    for m in METRICS:
        paired_rows.append({
            "comparison": "Expanded - Original",
            "fold": f"CV{cv}",
            "metric": m,
            "original_score": orig[m],
            "expanded_score": exp[m],
            "difference": exp[m] - orig[m],
        })

paired = pd.DataFrame(paired_rows)
paired.to_csv(os.path.join(OUT_DIR, "part1_fold_paired_differences_expanded_minus_original.csv"), index=False)

display(paired)

# -------------------------
# Difference mean ± SD across folds
# -------------------------
diff_summary_rows = []
for m, g in paired.groupby("metric"):
    mean = g["difference"].mean()
    sd = g["difference"].std(ddof=1)
    diff_summary_rows.append({
        "comparison": "Expanded - Original",
        "metric": m,
        "difference_mean": mean,
        "difference_sd": sd,
        "difference_mean_sd": f"{mean:+.4f} ± {sd:.4f}",
    })

diff_summary = pd.DataFrame(diff_summary_rows)
diff_summary = diff_summary.set_index("metric").loc[METRICS].reset_index()
diff_summary.to_csv(os.path.join(OUT_DIR, "part1_paired_difference_summary_expanded_minus_original.csv"), index=False)

display(diff_summary)

# -------------------------
# Compact manuscript/advisor table
# -------------------------
compact_rows = []
for condition in ["Original", "Expanded"]:
    s = summary[summary["condition"] == condition].iloc[0]
    compact_rows.append({
        "Condition": condition,
        "Fold scope": "5 folds",
        "Accuracy": s["accuracy_mean_sd"],
        "Macro F1": s["macro_f1_mean_sd"],
        "Weighted F1": s["weighted_f1_mean_sd"],
        "Chitchat F1": s["chitchat_f1_mean_sd"],
        "Interpret F1": s["interpret_f1_mean_sd"],
        "Reveal F1": s["reveal_f1_mean_sd"],
        "Revise F1": s["revise_f1_mean_sd"],
        "Inquire F1": s["inquire_f1_mean_sd"],
    })

compact = pd.DataFrame(compact_rows)
compact.to_csv(os.path.join(OUT_DIR, "part1_compact_5cv_table.csv"), index=False)

display(compact)

# -------------------------
# Optional: CV1 corrected fixed-weight sensitivity table
# -------------------------
corrected_path = os.path.join(CORRECTED_CV1_DIR, "test_metrics.csv")

if os.path.exists(corrected_path):
    orig_cv1 = fold_metrics[(fold_metrics["condition"] == "Original") & (fold_metrics["fold"] == "CV1")].iloc[0]
    exp_cv1 = fold_metrics[(fold_metrics["condition"] == "Expanded") & (fold_metrics["fold"] == "CV1")].iloc[0]
    corr_cv1 = pd.read_csv(corrected_path).iloc[0].to_dict()

    cv1_rows = []
    for name, row in [
        ("Original", orig_cv1),
        ("Expanded", exp_cv1),
        ("Corrected fixed Expanded weights", corr_cv1),
    ]:
        cv1_rows.append({
            "Condition": name,
            "Fold / test set": "CV1",
            "Accuracy": row["accuracy"],
            "Macro F1": row["macro_f1"],
            "Weighted F1": row["weighted_f1"],
            "Chitchat F1": row["chitchat_f1"],
            "Interpret F1": row["interpret_f1"],
            "Reveal F1": row["reveal_f1"],
            "Revise F1": row["revise_f1"],
            "Inquire F1": row["inquire_f1"],
        })

    cv1_table = pd.DataFrame(cv1_rows)
    cv1_table.to_csv(os.path.join(OUT_DIR, "part2_cv1_corrected_sensitivity_table.csv"), index=False)

    display(cv1_table)

print("Saved all Part 1/Part 2 table files to:")
print(OUT_DIR)

Mounted at /content/drive


,condition,fold,accuracy,macro_f1,weighted_f1,chitchat_f1,interpret_f1,reveal_f1,revise_f1,inquire_f1
0,Original,CV1,0.829688,0.459403,0.813825,0.600000,0.883117,0.813896,0.000000,0.0
1,Expanded,CV1,0.815625,0.445573,0.802677,0.540000,0.869908,0.817955,0.000000,0.0
2,Original,CV2,0.857590,0.459713,0.839431,0.518519,0.908377,0.871671,0.000000,0.0
3,Expanded,CV2,0.845070,0.459146,0.830693,0.543210,0.900388,0.852130,0.000000,0.0
4,Original,CV3,0.858934,0.485040,0.844849,0.659574,0.899598,0.866029,0.000000,0.0
5,Expanded,CV3,0.785266,0.499624,0.806303,0.691358,0.840941,0.832487,0.133333,0.0
6,Original,CV4,0.849765,0.472169,0.833459,0.627907,0.902471,0.830467,0.000000,0.0
7,Expanded,CV4,0.846635,0.502714,0.845884,0.643678,0.904509,0.854271,0.111111,0.0
8,Original,CV5,0.853125,0.478118,0.844167,0.617886,0.902703,0.870000,0.000000,0.0
9,Expanded,CV5,0.832812,0.516625,0.828921,0.653061,0.875000,0.848168,0.206897,0.0


,condition,completed_folds,accuracy_mean,accuracy_sd,accuracy_mean_sd,macro_f1_mean,macro_f1_sd,macro_f1_mean_sd,weighted_f1_mean,weighted_f1_sd,...,interpret_f1_mean_sd,reveal_f1_mean,reveal_f1_sd,reveal_f1_mean_sd,revise_f1_mean,revise_f1_sd,revise_f1_mean_sd,inquire_f1_mean,inquire_f1_sd,inquire_f1_mean_sd
0,Expanded,5,0.825082,0.025488,0.8251 ± 0.0255,0.484736,0.030620,0.4847 ± 0.0306,0.822896,0.018095,...,0.8781 ± 0.0257,0.841002,0.015451,0.8410 ± 0.0155,0.090268,0.089705,0.0903 ± 0.0897,0.0,0.0,0.0000 ± 0.0000
1,Original,5,0.849820,0.011829,0.8498 ± 0.0118,0.470889,0.011303,0.4709 ± 0.0113,0.835146,0.012761,...,0.8993 ± 0.0096,0.850412,0.026508,0.8504 ± 0.0265,0.000000,0.000000,0.0000 ± 0.0000,0.0,0.0,0.0000 ± 0.0000


,comparison,fold,metric,original_score,expanded_score,difference
0,Expanded - Original,CV1,accuracy,0.829688,0.815625,-0.014062
1,Expanded - Original,CV1,macro_f1,0.459403,0.445573,-0.013830
2,Expanded - Original,CV1,weighted_f1,0.813825,0.802677,-0.011148
3,Expanded - Original,CV1,chitchat_f1,0.600000,0.540000,-0.060000
4,Expanded - Original,CV1,interpret_f1,0.883117,0.869908,-0.013209
5,Expanded - Original,CV1,reveal_f1,0.813896,0.817955,0.004059
6,Expanded - Original,CV1,revise_f1,0.000000,0.000000,0.000000
7,Expanded - Original,CV1,inquire_f1,0.000000,0.000000,0.000000
8,Expanded - Original,CV2,accuracy,0.857590,0.845070,-0.012520
9,Expanded - Original,CV2,macro_f1,0.459713,0.459146,-0.000568


,metric,comparison,difference_mean,difference_sd,difference_mean_sd
0,accuracy,Expanded - Original,-0.024738,0.028035,-0.0247 ± 0.0280
1,macro_f1,Expanded - Original,0.013848,0.021571,+0.0138 ± 0.0216
2,weighted_f1,Expanded - Original,-0.012251,0.018186,-0.0123 ± 0.0182
3,chitchat_f1,Expanded - Original,0.009484,0.039548,+0.0095 ± 0.0395
4,interpret_f1,Expanded - Original,-0.021104,0.023578,-0.0211 ± 0.0236
5,reveal_f1,Expanded - Original,-0.009410,0.023043,-0.0094 ± 0.0230
6,revise_f1,Expanded - Original,0.090268,0.089705,+0.0903 ± 0.0897
7,inquire_f1,Expanded - Original,0.000000,0.000000,+0.0000 ± 0.0000


,Condition,Fold scope,Accuracy,Macro F1,Weighted F1,Chitchat F1,Interpret F1,Reveal F1,Revise F1,Inquire F1
0,Original,5 folds,0.8498 ± 0.0118,0.4709 ± 0.0113,0.8351 ± 0.0128,0.6048 ± 0.0529,0.8993 ± 0.0096,0.8504 ± 0.0265,0.0000 ± 0.0000,0.0000 ± 0.0000
1,Expanded,5 folds,0.8251 ± 0.0255,0.4847 ± 0.0306,0.8229 ± 0.0181,0.6143 ± 0.0687,0.8781 ± 0.0257,0.8410 ± 0.0155,0.0903 ± 0.0897,0.0000 ± 0.0000


,Condition,Fold / test set,Accuracy,Macro F1,Weighted F1,Chitchat F1,Interpret F1,Reveal F1,Revise F1,Inquire F1
0,Original,CV1,0.829688,0.459403,0.813825,0.600000,0.883117,0.813896,0.0,0.0
1,Expanded,CV1,0.815625,0.445573,0.802677,0.540000,0.869908,0.817955,0.0,0.0
2,Corrected fixed Expanded weights,CV1,0.820312,0.457914,0.808060,0.606061,0.875325,0.808184,0.0,0.0


Saved all Part 1/Part 2 table files to:
/content/drive/MyDrive/thesis_intent_corrected code_august2026_crossvalidations(4and5 class)/5cv_5class_intent_12Aug/results/part1_original_vs_expanded_paired_tables
